# Perhitungan Volume Kedua Lambung Kapal

**Analisis Numerik**

Ketentuan:
- 1 dash dan 1 ruang kosong pada grid = 5 + 5 cm = 10 cm atau 0,1 m.
- Lantai kapal dianggap datar.
- Tampak depan dianggap berbentuk persegi panjang.
- Volume reserve buoyancy tidak dihitung.
- Volume kedua lambung kapal dihitung menggunakan integrasi numerik.

## 1. Import Library dan Menampilkan Gambar

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

In [ ]:
# Nama file gambar
file_gambar = "Kapal (1).png"

# Membaca gambar
kapal = mpimg.imread(file_gambar)

# Menampilkan gambar
plt.figure(figsize=(15, 8))
plt.imshow(kapal)
plt.title("Gambar Perahu Katamaran dengan Grid")
plt.xlabel("Koordinat X")
plt.ylabel("Koordinat Y")
plt.axis("on")
plt.show()

## 2. Menentukan Titik Pengukuran

In [ ]:
# Banyak interval dan jarak antar titik
jumlah_interval = 16
jarak = 0.1  # meter

# Membuat 17 titik dari 0 sampai 1,6 meter
posisi_x = np.linspace(0, jumlah_interval * jarak, jumlah_interval + 1)

# Nama titik
nama_titik = [f"x{i}" for i in range(len(posisi_x))]

# Membuat tabel titik
titik_pengukuran = pd.DataFrame({
    "Titik": nama_titik,
    "Posisi (m)": posisi_x
})

titik_pengukuran

## 3. Menentukan Posisi Titik pada Gambar

In [ ]:
# Koordinat pixel titik pengukuran
x_awal = 115
jarak_pixel = 40

koordinat_pixel = [x_awal + i * jarak_pixel for i in range(17)]

plt.figure(figsize=(15, 8))
plt.imshow(kapal)

for nomor, posisi in enumerate(koordinat_pixel):
    plt.axvline(posisi, linestyle="--", linewidth=1)
    plt.text(
        posisi, 25, f"x{nomor}", ha="center", fontsize=9,
        bbox={"facecolor": "white", "alpha": 0.8}
    )

plt.title("Titik Pengukuran Lambung Kapal")
plt.xlabel("Posisi horizontal (pixel)")
plt.ylabel("Posisi vertikal (pixel)")
plt.show()

## 4. Menentukan Ukuran Penampang

Lebar lambung ditentukan pada setiap titik pengukuran. Tinggi lambung dianggap tetap sebesar 25,50 cm sesuai asumsi lantai kapal datar.

In [ ]:
# Posisi pengukuran dalam cm
posisi_cm = np.arange(0, 161, 10)

# Lebar lambung pada setiap titik (cm)
lebar_cm = np.array([
    18.00, 19.50, 21.00, 21.25, 21.50,
    21.50, 21.50, 21.50, 21.50, 21.50,
    21.50, 21.50, 21.00, 20.50, 19.00,
    16.75, 13.75
])

# Tinggi lambung dibuat sama pada seluruh titik
tinggi_cm = np.repeat(25.50, 17)

# Luas penampang dalam cm²
luas_cm2 = lebar_cm * tinggi_cm

# Konversi ke meter
posisi_m = posisi_cm / 100
lebar_m = lebar_cm / 100
tinggi_m = tinggi_cm / 100
luas_m2 = luas_cm2 / 10000

In [ ]:
hasil_penampang = pd.DataFrame({
    "Titik": nama_titik,
    "Posisi (m)": posisi_m,
    "Lebar (m)": lebar_m,
    "Tinggi (m)": tinggi_m,
    "Luas Penampang (m²)": luas_m2
})

hasil_penampang

## 5. Menghitung Volume Satu Lambung dengan Metode Simpson 1/3

Rumus:

$$V = \frac{h}{3}[A_0 + A_n + 4(A_1+A_3+\cdots+A_{15}) + 2(A_2+A_4+\cdots+A_{14})]$$

In [ ]:
h = 0.1

A_awal = luas_m2[0]
A_akhir = luas_m2[-1]
jumlah_ganjil = np.sum(luas_m2[1:-1:2])
jumlah_genap = np.sum(luas_m2[2:-1:2])

volume_simpson = (h / 3) * (
    A_awal + A_akhir +
    4 * jumlah_ganjil +
    2 * jumlah_genap
)

print("HASIL METODE SIMPSON 1/3")
print("-" * 35)
print(f"A0              : {A_awal:.6f} m²")
print(f"A16             : {A_akhir:.6f} m²")
print(f"Jumlah A ganjil : {jumlah_ganjil:.6f} m²")
print(f"Jumlah A genap  : {jumlah_genap:.6f} m²")
print(f"Volume           : {volume_simpson:.6f} m³")
print(f"Volume           : {volume_simpson * 1000:.2f} liter")

## 6. Menghitung Volume Kedua Lambung

In [ ]:
banyak_lambung = 2
volume_simpson_total = volume_simpson * banyak_lambung

print("HASIL VOLUME KEDUA LAMBUNG")
print("-" * 35)
print(f"Satu lambung : {volume_simpson:.6f} m³")
print(f"Dua lambung : {volume_simpson_total:.6f} m³")
print(f"Dua lambung : {volume_simpson_total * 1000:.2f} liter")

## 7. Menghitung Volume dengan Metode Trapesium

In [ ]:
volume_trapesium = np.trapezoid(luas_m2, posisi_m)
volume_trapesium_total = volume_trapesium * banyak_lambung

print("HASIL METODE TRAPESIUM")
print("-" * 35)
print(f"Satu lambung : {volume_trapesium:.6f} m³")
print(f"Dua lambung : {volume_trapesium_total:.6f} m³")
print(f"Dua lambung : {volume_trapesium_total * 1000:.2f} liter")

## 8. Perbandingan Hasil Kedua Metode

In [ ]:
selisih_volume = abs(volume_simpson - volume_trapesium)
persentase = (selisih_volume / volume_simpson) * 100
selisih_total = abs(volume_simpson_total - volume_trapesium_total)

print("PERBANDINGAN HASIL")
print("=" * 45)
print(f"Simpson 1/3 : {volume_simpson:.9f} m³")
print(f"Trapesium   : {volume_trapesium:.9f} m³")
print(f"Selisih     : {selisih_volume:.9f} m³")
print(f"Persentase  : {persentase:.3f}%")

print("\nUntuk dua lambung:")
print(f"Simpson 1/3 : {volume_simpson_total:.9f} m³")
print(f"Trapesium   : {volume_trapesium_total:.9f} m³")
print(f"Selisih     : {selisih_total:.9f} m³")
print(f"Selisih     : {selisih_total * 1000:.3f} liter")

## 9. Tabel Perbandingan

In [ ]:
tabel_hasil = pd.DataFrame({
    "Metode": ["Simpson 1/3", "Trapesium"],
    "Volume 1 Lambung (m³)": [volume_simpson, volume_trapesium],
    "Volume 2 Lambung (m³)": [volume_simpson_total, volume_trapesium_total],
    "Volume 1 Lambung (Liter)": [volume_simpson * 1000, volume_trapesium * 1000],
    "Volume 2 Lambung (Liter)": [volume_simpson_total * 1000, volume_trapesium_total * 1000]
})

tabel_hasil

## Kesimpulan

Berdasarkan perhitungan integrasi numerik, volume kedua lambung kapal diperoleh sekitar **0,166898 m³ atau 166,90 liter** menggunakan metode Simpson 1/3. Metode Trapesium menghasilkan sekitar **0,166706 m³ atau 166,71 liter**.

Selisih kedua metode relatif kecil, yaitu sekitar **0,000191 m³ atau 0,191 liter**, sehingga kedua metode memberikan hasil yang berdekatan.